In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

# 1) TSV 로드 ─ 경로만 맞춰 주세요
file_path = "datasets/result/all_performance.tsv"
df = pd.read_csv(file_path, sep="\t")

# 2) Ours / Baseline 행만 추려서 사용
df = df[df["method"].isin(["Ours", "Baseline"])]

turn_cols   = ["turn_2", "turn_3", "turn_4", "turn_5"]
turn_labels = [2, 3, 4, 5]

# 3) 모델 고유색 팔레트 (Tab10 순환)
models      = sorted(df["model"].unique())
cmap        = get_cmap("tab10")
model_color = {m: cmap(i % 10) for i, m in enumerate(models)}

plt.figure(figsize=(12, 6))

for model in models:
    sub = df[df["model"] == model]

    ours     = sub[sub["method"] == "Ours"]
    baseline = sub[sub["method"] == "Baseline"]

    # --- Ours: 고유색 · 실선 · 원형 마커
    if not ours.empty:
        plt.plot(
            turn_labels,
            ours[turn_cols].values.flatten(),
            marker="o",
            linestyle="-",
            linewidth=2.2,
            color=model_color[model],
            label=f"{model} – Ours",
        )

    # --- Baseline: 회색 · 점선 · 네모 마커
    if not baseline.empty:
        plt.plot(
            turn_labels,
            baseline[turn_cols].values.flatten(),
            marker="s",
            linestyle="--",
            linewidth=1.2,
            color="gray",
            alpha=0.6,
            label=f"{model} – Baseline",
        )

plt.title("Multi-turn Accuracy Comparison (Turn 2→5)")
plt.xlabel("Turn")
plt.ylabel("Accuracy (%)")
plt.xticks(turn_labels)
plt.grid(axis="y", linestyle=":")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

# Load TSV
file_path = "datasets/result/all_performance.tsv"
df = pd.read_csv(file_path, sep="\t")

# Keep only Ours and Baseline
df = df[df["method"].isin(["Ours", "Baseline"])]

turn_cols   = ["turn_2", "turn_3", "turn_4", "turn_5"]
turn_labels = [2, 3, 4, 5]

# Desired subplot order
ordered_models = [
    "o4-mini",
    "gpt41",
    "gpt-4o-mini",
    "gemini-2.0-flash",
    "llama3-3b",
    "phi-4-4b",
    "qwen2.5-3b",
    "qwen3-4b",
]

# Filter out any model in list that might be missing in the data
models = [m for m in ordered_models if m in df["model"].unique()]

# Color palette
cmap = get_cmap("tab10")
model_color = {m: cmap(i % 10) for i, m in enumerate(models)}

# Unified y‑axis limits
y_min = df[turn_cols].min().min() - 1
y_max = df[turn_cols].max().max() + 1

# 2×4 grid
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharex=True, sharey=True)
axes = axes.flatten()

for idx, model in enumerate(models):
    ax = axes[idx]
    sub = df[df["model"] == model]
    
    ours     = sub[sub["method"] == "Ours"]
    baseline = sub[sub["method"] == "Baseline"]
    
    if not baseline.empty:
        ax.plot(
            turn_labels,
            baseline[turn_cols].values.flatten(),
            marker="s",
            linestyle="--",
            linewidth=1.5,
            color="gray",
            alpha=0.7,
            label="Baseline" if idx == 0 else None,
        )
    if not ours.empty:
        ax.plot(
            turn_labels,
            ours[turn_cols].values.flatten(),
            marker="o",
            linestyle="-",
            linewidth=2.5,
            color=model_color[model],
            label="Ours" if idx == 0 else None,
        )
    
    ax.set_title(model, fontsize=10)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(turn_labels)
    if idx >= 4:
        ax.set_xlabel("Turn", fontsize=9)
    if idx % 4 == 0:
        ax.set_ylabel("Accuracy (%)", fontsize=9)
    ax.grid(axis="y", linestyle=":", alpha=0.5)

# Hide any unused axes
for j in range(len(models), len(axes)):
    axes[j].axis("off")

# Global legend at bottom center
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, ["Baseline", "Ours"], loc="lower center",
           ncol=2, frameon=False, fontsize=10, bbox_to_anchor=(0.5, -0.05))

# Tight layout with extra bottom space for legend
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

# Load TSV
file_path = "datasets/result/all_performance.tsv"
df = pd.read_csv(file_path, sep="\t")

turn_cols   = ["turn_2", "turn_3", "turn_4", "turn_5"]
turn_labels = [2, 3, 4, 5]

ours_models = ["o4-mini", "GPT-4.1", "gpt-4o-mini", "gemini-2.0-flash"]
rma_models  = ["llama3-3b", "phi-4-mini-4b", "qwen2.5-3b", "qwen3-4b"]

ordered_models = ours_models + rma_models
models = [m for m in ordered_models if m in df["model"].unique()]

cmap = get_cmap("tab10")
model_color = {m: cmap(i % 10) for i, m in enumerate(models)}

y_min = df[turn_cols].min().min() - 1
y_max = df[turn_cols].max().max() + 1

fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharex=True, sharey=True)
axes = axes.flatten()

label_used = {"Baseline": False, "Ideal Rewrite": False, "RMA": False}

for idx, model in enumerate(models):
    ax = axes[idx]
    sub = df[df["model"] == model]

    # Baseline
    baseline = sub[sub["method"] == "Baseline"]
    if not baseline.empty:
        ax.plot(turn_labels, baseline[turn_cols].values.flatten(),
                marker="s", linestyle="--", linewidth=1.5,
                color="gray", alpha=0.7,
                label="Baseline" if not label_used["Baseline"] else None)
        label_used["Baseline"] = True

    # Ours -> Ideal Rewrite
    ours_line = sub[sub["method"] == "Ours"]
    if not ours_line.empty:
        ax.plot(turn_labels, ours_line[turn_cols].values.flatten(),
                marker="o", linestyle="-", linewidth=2.5,
                color=model_color[model],
                label="Ideal Rewrite" if not label_used["Ideal Rewrite"] else None)
        label_used["Ideal Rewrite"] = True

    # RMA for rma_models
    if model in rma_models:
        rma_line = sub[sub["method"] == "RMA"]
        if not rma_line.empty:
            ax.plot(turn_labels, rma_line[turn_cols].values.flatten(),
                    marker="^", linestyle=":", linewidth=2.0,
                    color=model_color[model],
                    label="RMA" if not label_used["RMA"] else None)
            label_used["RMA"] = True

    ax.set_title(model, fontsize=10)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(turn_labels)
    if idx >= 4:
        ax.set_xlabel("Turn", fontsize=9)
    if idx % 4 == 0:
        ax.set_ylabel("Accuracy (%)", fontsize=9)
    ax.grid(axis="y", linestyle=":", alpha=0.5)

# hide unused axes
for j in range(len(models), len(axes)):
    axes[j].axis("off")

# Collect unique legend handles
handles_labels = []
for ax in axes:
    handles_labels += list(zip(*ax.get_legend_handles_labels()))
handles_dict = {lab: h for h, lab in handles_labels}
legend_order = ["Baseline", "Ideal Rewrite", "RMA"]
handles = [handles_dict[l] for l in legend_order if l in handles_dict]

fig.legend(handles, legend_order, loc="lower center",
           ncol=len(handles), frameon=False, fontsize=10, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()
# Save the current figure as a single-page PDF
fig.savefig("all_performance.pdf", format="pdf", bbox_inches="tight")  # dpi is ignored for vector formats

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

# --------------------------- 1. Load data ---------------------------
file_path = "datasets/result/all_performance.tsv"
df = pd.read_csv(file_path, sep="\t")

turn_cols   = ["turn_2", "turn_3", "turn_4", "turn_5"]
turn_labels = [2, 3, 4, 5]

ours_models = ["o4-mini", "GPT-4.1", "gpt-4o-mini", "gemini-2.0-flash"]
rma_models  = ["llama3-3b", "phi-4-mini-4b", "qwen2.5-3b", "qwen3-4b"]

ordered_models = ours_models + rma_models
#models = [m for m in ordered_models if m in df["model"].unique()]
models = [m for m in MODEL_ORDER if m in df["model"].unique()]
# --------------------------- 2. Style ---------------------------
BIG_TITLE   = 20   # per-subplot title
AXIS_LABEL  = 25   # x / y label
TICK_LABEL  = 11   # tick-label
LEGEND_FONT = 25   # legend

plt.rcParams.update({
    "axes.titlesize": BIG_TITLE,
    "axes.labelsize": AXIS_LABEL,
    "xtick.labelsize": TICK_LABEL,
    "ytick.labelsize": TICK_LABEL,
    "legend.fontsize": LEGEND_FONT,
})

cmap = get_cmap("tab10")
model_color = {m: cmap(i % 10) for i, m in enumerate(models)}

y_min = df[turn_cols].min().min() - 1
y_max = df[turn_cols].max().max() + 1

fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharex=True, sharey=True)
axes = axes.flatten()

label_used = {"Baseline": False, "Ideal Rewrite": False, "RMA": False}

for idx, model in enumerate(models):
    ax = axes[idx]
    sub = df[df["model"] == model]

    # Baseline
    baseline = sub[sub["method"] == "Baseline"]
    if not baseline.empty:
        ax.plot(turn_labels, baseline[turn_cols].values.flatten(),
                marker="s", linestyle="--", linewidth=1.5,
                color="gray", alpha=0.7,
                label="Baseline" if not label_used["Baseline"] else None)
        label_used["Baseline"] = True

    # Ours -> Ideal Rewrite
    ours_line = sub[sub["method"] == "Ours"]
    if not ours_line.empty:
        ax.plot(turn_labels, ours_line[turn_cols].values.flatten(),
                marker="o", linestyle="-", linewidth=2.5,
                color=MODEL_COLOR[model],#color=model_color[model],
                label="Ideal Rewrite" if not label_used["Ideal Rewrite"] else None)
        label_used["Ideal Rewrite"] = True

    # RMA for rma_models
    if model in rma_models:
        rma_line = sub[sub["method"] == "RMA"]
        if not rma_line.empty:
            ax.plot(turn_labels, rma_line[turn_cols].values.flatten(),
                    marker="^", linestyle=":", linewidth=2.0,
                    color=MODEL_COLOR[model],#color=model_color[model],
                    label="RMA" if not label_used["RMA"] else None)
            label_used["RMA"] = True

    ax.set_title(model)                       # ← BIG_TITLE 적용
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(turn_labels)
    if idx >= 4:
        ax.set_xlabel("Turn")                 # ← AXIS_LABEL 적용
    #if idx % 4 == 0:
    #    ax.set_ylabel("Accuracy (%)")         # ← AXIS_LABEL 적용
    ax.grid(axis="y", linestyle=":", alpha=0.5)

fig.supylabel("Accuracy (%)", x=0.02, y=0.5, ha="center",
              va="center", rotation="vertical", fontsize=AXIS_LABEL)

# hide unused axes
for j in range(len(models), len(axes)):
    axes[j].axis("off")

# --------------------------- 3. Legend ---------------------------
handles_labels = []
for ax in axes:
    handles_labels += list(zip(*ax.get_legend_handles_labels()))
handles_dict = {lab: h for h, lab in handles_labels}

legend_order = ["Baseline", "Ideal Rewrite", "RMA"]
legend_order = ["Ideal Rewrite", "RMA", "Baseline"]
handles = [handles_dict[l] for l in legend_order if l in handles_dict]

fig.legend(handles, legend_order, loc="lower center",
           ncol=len(handles), frameon=False,
           bbox_to_anchor=(0.5, -0.06))       # LEGEND_FONT은 rcParams에서 자동 적용

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

# Save as vector PDF (embeds fonts cleanly)
fig.savefig("overall_comparison.pdf", format="pdf", bbox_inches="tight")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

# --------------------------- 1. Load data ---------------------------
file_path = "datasets/result/all_performance.tsv"
df = pd.read_csv(file_path, sep="\t")

turn_cols   = ["turn_2", "turn_3", "turn_4", "turn_5"]
turn_labels = [2, 3, 4, 5]

ours_models = ["o4-mini", "GPT-4.1", "GPT-4o-mini", "Gemini-2.0-flash"]
rma_models  = ["LLaMA-3.2 (3B)", "Phi-4-mini (4B)", "Qwen2.5 (3B)", "Qwen3 (4B)"]

ordered_models = ours_models + rma_models
models = [m for m in ordered_models if m in df["model"].unique()]

# --------------------------- 2. Style ---------------------------
BIG_TITLE   = 20   # per-subplot title
AXIS_LABEL  = 24   # x / y label
TICK_LABEL  = 20   # tick label
LEGEND_FONT = 27   # legend
TURN_LABEL_FONT  = 20

plt.rcParams.update({
    "axes.titlesize": BIG_TITLE,
    "axes.labelsize": AXIS_LABEL,
    "xtick.labelsize": TICK_LABEL,
    "ytick.labelsize": TICK_LABEL,
    "legend.fontsize": LEGEND_FONT,
})

# (선호 색상 있으면 여기서 바꿔도 됨)
cmap = get_cmap("tab10")
method_color = {
    "Baseline":        "gray",          # 고정 회색
    "RMA (Oracle Rewrite)":   cmap(0),         # 파란 계열
    "RMA":             cmap(1),         # 주황 계열
}

y_min = df[turn_cols].min().min() - 1
y_max = df[turn_cols].max().max() + 1

fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharex=True, sharey=True)
axes = axes.flatten()

label_used = {"Baseline": False, "RMA (Oracle Rewrite)": False, "RMA": False}

for idx, model in enumerate(models):
    ax = axes[idx]
    sub = df[df["model"] == model]

    # Baseline
    baseline = sub[sub["method"] == "Baseline"]
    if not baseline.empty:
        ax.plot(
            turn_labels, baseline[turn_cols].values.flatten(),
            marker="s", linestyle="--", linewidth=1.5,
            color=method_color["Baseline"], alpha=0.7,
            label="Baseline" if not label_used["Baseline"] else None
        )
        label_used["Baseline"] = True

    # RMA (Oracle Rewrite)
    ideal = sub[sub["method"] == "Ours"]
    if not ideal.empty:
        ax.plot(
            turn_labels, ideal[turn_cols].values.flatten(),
            marker="o", linestyle="-", linewidth=2.5,
            color=method_color["RMA (Oracle Rewrite)"],
            label="RMA (Oracle Rewrite)" if not label_used["RMA (Oracle Rewrite)"] else None
        )
        label_used["RMA (Oracle Rewrite)"] = True

    # RMA (해당 모델이 rma_models 집합에 있을 때만)
    if model in rma_models:
        rma = sub[sub["method"] == "RMA"]
        if not rma.empty:
            ax.plot(
                turn_labels, rma[turn_cols].values.flatten(),
                marker="^", linestyle=":", linewidth=2.0,
                color=method_color["RMA"],
                label="RMA" if not label_used["RMA"] else None
            )
            label_used["RMA"] = True

    ax.set_title(model)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(turn_labels)
    ax.tick_params(axis="x", labelsize=TURN_LABEL_FONT)
    if idx >= 4:
        ax.set_xlabel("Turn")
    ax.grid(axis="y", linestyle=":", alpha=0.5)

# 공통 y-label
fig.supylabel("Accuracy (%)", x=0.02, y=0.5, ha="center",
              va="center", rotation="vertical", fontsize=AXIS_LABEL)

# 미사용 서브플롯 숨기기
for j in range(len(models), len(axes)):
    axes[j].axis("off")

# --------------------------- 3. Legend ---------------------------
handles_labels = []
for ax in axes:
    handles_labels += list(zip(*ax.get_legend_handles_labels()))
handles_dict = {lab: h for h, lab in handles_labels}

legend_order = ["RMA (Oracle Rewrite)", "RMA", "Baseline"]
handles = [handles_dict[l] for l in legend_order if l in handles_dict]

fig.legend(
    handles, legend_order, loc="lower center",
    ncol=len(handles), frameon=False,
    bbox_to_anchor=(0.5, -0.06)
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

# Save as vector PDF (embeds fonts cleanly)
fig.savefig("overall_comparison.pdf", format="pdf", bbox_inches="tight")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
from matplotlib.lines import Line2D
from matplotlib.colors import to_rgb, to_hex

# 1. 데이터 하나만 로드
df = pd.read_csv("datasets/result/new2_all_performance.tsv", sep="\t")

turn_cols   = ["turn_2", "turn_3", "turn_4", "turn_5"]
turn_labels = [2, 3, 4, 5]
ordered_models = [
    "o4-mini", "GPT-4.1", "GPT-4o-mini", "Gemini-2.0-flash",
    "LLaMA-3.2 (3B)", "Phi-4-mini (4B)", "Qwen2.5 (3B)", "Qwen3 (4B)"
]
models = [m for m in ordered_models if m in df["model"].unique()]

# 2. 색상 헬퍼
cmap = get_cmap("tab10")
base_color = {"Baseline":"gray",
              "RMA (Oracle Rewrite)": cmap(0),
              "RMA": cmap(1)}

def lighten(hex_c, r=0.55):
    r0,g0,b0 = to_rgb(hex_c); return to_hex((r0+(1-r0)*r, g0+(1-g0)*r, b0+(1-b0)*r))
def darken (hex_c, r=0.25):
    r0,g0,b0 = to_rgb(hex_c); return to_hex((r0*(1-r), g0*(1-r), b0*(1-r)))

def canonical(m):  # 'Ours' → 공식 명칭
    return "RMA (Oracle Rewrite)" if m.replace("+","").strip()=="Ours" else m.replace("+","").strip()

def pick_color(m):
    return lighten(base_color[canonical(m)]) if m.endswith("+") else darken(base_color[canonical(m)])

# 3. 스타일
plt.rcParams.update({"axes.titlesize":20,"axes.labelsize":24,
                     "xtick.labelsize":20,"ytick.labelsize":20,
                     "legend.fontsize":22})

y_min = df[turn_cols].min().min() - 1
y_max = df[turn_cols].max().max() + 1
fig, axes = plt.subplots(2,4, figsize=(16,8), sharex=True, sharey=True)
axes = axes.flatten()

legend_seen = set()
for idx, model in enumerate(models):
    ax = axes[idx]; sub = df[df["model"]==model]

    for raw, mk, ls in [("Baseline","s","--"),
                        ("Ours","o","-"),
                        ("RMA","^",":")]:
        row = sub[sub["method"].str.startswith(raw)]
        if row.empty: continue
        m_str  = row["method"].iloc[0]
        label  = canonical(m_str)+('+' if m_str.endswith('+') else '')
        color  = pick_color(m_str)
        y_vals = row.iloc[0][turn_cols].values
        ax.plot(turn_labels, y_vals, marker=mk, linestyle=ls, linewidth=2,
                color=color, label=label if label not in legend_seen else None)
        legend_seen.add(label)

    ax.set_title(model)
    ax.set_ylim(y_min,y_max)
    ax.set_xticks(turn_labels)
    if idx>=4: ax.set_xlabel("Turn")
    ax.grid(axis="y", linestyle=":", alpha=0.5)

fig.supylabel("Accuracy (%)", x=0.02, rotation="vertical")

for j in range(len(models), len(axes)):
    axes[j].axis("off")

# 4. 두 줄 범례
handles, labels = [], []
for ax in axes: h,l = ax.get_legend_handles_labels(); handles+=h; labels+=l
hdict = {lab:h for h,lab in zip(handles,labels)}
top    = ["RMA (Oracle Rewrite)", "Baseline"]
bottom = ["RMA (Oracle Rewrite)+", "RMA+", "Baseline+"]

fig.legend([hdict[t] for t in top], top,
           loc="lower center", ncol=len(top), frameon=False,
           bbox_to_anchor=(0.5,-0.06))
fig.legend([hdict[b] for b in bottom if b in hdict],
           [b for b in bottom if b in hdict],
           loc="lower center", ncol=5, frameon=False,
           bbox_to_anchor=(0.5,-0.12))

plt.tight_layout(rect=[0,0.05,1,1]); plt.show()
fig.savefig("overall_comparison.pdf", format="pdf", bbox_inches="tight")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

# 1. 데이터
df = pd.read_csv("datasets/result/new2_all_performance.tsv", sep="\t")
turn_cols   = ["turn_2", "turn_3", "turn_4", "turn_5"]
turn_labels = [2,3,4,5]
models = [
    "o4-mini","GPT-4.1","GPT-4o-mini","Gemini-2.0-flash",
    "LLaMA-3.2 (3B)","Phi-4-mini (4B)","Qwen2.5 (3B)","Qwen3 (4B)"
]

# 2. 팔레트 ── 진·연 파랑 / 주황 교차
cmap = get_cmap("tab10")           # cmap(0)=진파랑, cmap(1)=진주황
palette = {
    # Baseline 계열
    "Baseline"  : "#bdbdbd",       # 연회색
    "Baseline+" : "#525252",       # 진회색

    # RMA (Oracle Rewrite) 계열
    "Ours"                       : "#9ec2ff",  # 연파랑   (non-plus)
    "Ours+"                      : cmap(0),    # 진파랑   (plus)

    # RMA 계열
    "RMA"                        : "#ffb55a",  # 연주황   (non-plus)
    "RMA+"                       : "#d95f0e",  # 진주황   (plus)
}

def canonical(m):                             # 'Ours' → RMA (Oracle Rewrite)
    return "RMA (Oracle Rewrite)" if m.replace("+","").strip() == "Ours" else m.replace("+","").strip()

def pick_color(m):                            # 팔레트→ canonical 순으로 조회
    return palette.get(m, palette.get(canonical(m)))

# 3. 스타일
plt.rcParams.update({"axes.titlesize":20,"axes.labelsize":24,
                     "xtick.labelsize":20,"ytick.labelsize":20,
                     "legend.fontsize":22})
y_min = df[turn_cols].min().min()-1
y_max = df[turn_cols].max().max()+1
fig, axes = plt.subplots(2,4, figsize=(16,8), sharex=True, sharey=True)
axes = axes.flatten()

legend_seen=set()
plot_order = [                   # 정확 일치 6개 순회
    ("Baseline+","s","--"), ("Baseline","s","--"),
    ("Ours+","o","-"),      ("Ours","o","-"),
    ("RMA+","^",":"),       ("RMA","^",":")
]

for idx, model in enumerate(models):
    ax = axes[idx]; sub = df[df["model"]==model]
    for meth,mk,ls in plot_order:
        row = sub[sub["method"]==meth]
        if row.empty: continue
        label = canonical(meth)+("+" if meth.endswith("+") else "")
        ax.plot(turn_labels, row.iloc[0][turn_cols], marker=mk, linestyle=ls, linewidth=2,
                color=pick_color(meth),
                label=label if label not in legend_seen else None)
        legend_seen.add(label)

    ax.set_title(model)
    ax.set_xticks(turn_labels)
    ax.set_ylim(y_min,y_max)
    if idx>=4: ax.set_xlabel("Turn")
    ax.grid(axis="y", linestyle=":", alpha=.5)

fig.supylabel("Accuracy (%)", x=0.02, rotation="vertical")

# 빈 축 숨김
for j in range(len(models), len(axes)):
    axes[j].axis("off")

# 4. 범례 (두 줄)
h,l=[],[]
for ax in axes: hh,ll=ax.get_legend_handles_labels(); h+=hh; l+=ll
hdict={lab:handle for handle,lab in zip(h,l)}

top=["RMA (Oracle Rewrite)","Baseline"]
bottom=["RMA (Oracle Rewrite)+","RMA+","Baseline+"]

fig.legend([hdict[t] for t in top], top, loc="lower center",
           ncol=len(top), frameon=False, bbox_to_anchor=(0.5,-0.06))
fig.legend([hdict[b] for b in bottom], bottom, loc="lower center",
           ncol=len(bottom), frameon=False, bbox_to_anchor=(0.5,-0.12))

plt.tight_layout(rect=[0,0.05,1,1]); plt.show()
fig.savefig("overall_comparison.pdf", bbox_inches="tight")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

# ── 1. Load ───────────────────────────────────────────────
df = pd.read_csv("datasets/result/new_all_performance.tsv", sep="\t")

turn_cols   = ["turn_2", "turn_3", "turn_4", "turn_5"]
turn_labels = [2, 3, 4, 5]

closed = ["o4-mini", "GPT-4.1", "GPT-4o-mini", "Gemini-2.0-flash"]
opens  = ["LLaMA-3.2 (3B)", "Phi-4-mini (4B)", "Qwen2.5 (3B)", "Qwen3 (4B)"]

# ── 2. Palette ────────────────────────────────────────────
cmap = get_cmap("tab10")           # tab10[0] = 진파랑
palette = {
    # Baseline
    "Baseline"   : "#bdbdbd",
    "Baseline+"  : "#525252",
    # RMA (Oracle Rewrite) (=Ours)
    "Ours"       : "#9ec2ff",
    "Ours+"      : cmap(0),
    # RMA
    "RMA"        : "#ffb55a",
    "RMA+"       : "#d95f0e",
}

def pick_color(method: str) -> str:
    return palette.get(method, "#999999")

def pretty_label(method: str) -> str:
    if method.startswith("Ours"):
        base = "RMA (Oracle Rewrite)"
    elif method.startswith("Baseline"):
        base = "Baseline"
    elif method.startswith("RMA"):
        base = "RMA"
    else:
        base = method
    return base + ("+" if method.endswith("+") else "")

# ── 3. y-축 범위 (두 그룹) ─────────────────────────────────
mask_top = (df["model"].isin(closed)) | (df["method"].str.endswith("+"))
ymin_top = df.loc[mask_top, turn_cols].min().min() - 1
ymax_top = df.loc[mask_top, turn_cols].max().max() + 1

mask_bot = (df["model"].isin(opens)) & (~df["method"].str.endswith("+"))
ymin_bot = df.loc[mask_bot, turn_cols].min().min() - 1
ymax_bot = df.loc[mask_bot, turn_cols].max().max() + 1

# ── 4. Matplotlib 스타일 ─────────────────────────────────
plt.rcParams.update({
    "axes.titlesize":24, "axes.labelsize":24,
    "xtick.labelsize":14,"ytick.labelsize":20,
    "legend.fontsize":24})

fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharex=True, sharey=False)
axes = axes.flatten()
ax = lambda r, c: axes[r*4 + c]

legend_seen = set()
plot_order = [
    ("Baseline+", "s", "--"), ("Baseline", "s", "--"),
    ("Ours+"    , "o", "-"),  ("Ours",     "o", "-"),
    ("RMA+"     , "^", ":"),  ("RMA",      "^", ":")
]

def draw(ax_obj, subset, plus_filter=None):
    for meth, mk, ls in plot_order:
        if plus_filter is True  and not meth.endswith("+"):
            continue
        if plus_filter is False and  meth.endswith("+"):
            continue
        row = subset[subset["method"] == meth]
        if row.empty: continue
        label = pretty_label(meth)
        ax_obj.plot(turn_labels, row.iloc[0][turn_cols],
                    marker=mk, linestyle=ls, linewidth=2,
                    color=pick_color(meth),
                    label=label if label not in legend_seen else None)
        legend_seen.add(label)

# Row 0 – closed
for c, m in enumerate(closed):
    a = ax(0, c); a.set_title(m)
    draw(a, df[df["model"] == m], plus_filter=None)
    a.set_ylim(ymin_top, ymax_top); a.grid(axis="y", ls=":", alpha=.4)

# Row 1 – open plus
for c, m in enumerate(opens):
    a = ax(1, c); a.set_title(f"{m} (+)")
    draw(a, df[(df["model"] == m) & (df["method"].str.endswith("+"))], plus_filter=True)
    a.set_ylim(ymin_top, ymax_top); a.grid(axis="y", ls=":", alpha=.4)

# Row 2 – open non-plus
for c, m in enumerate(opens):
    a = ax(2, c); a.set_title(m)
    draw(a, df[(df["model"] == m) & (~df["method"].str.endswith("+"))], plus_filter=False)
    a.set_ylim(ymin_bot, ymax_bot); a.grid(axis="y", ls=":", alpha=.4)

# 공통 축
for r in range(3):
    for c in range(4):
        a = ax(r, c)
        a.set_xticks(turn_labels)
        if r == 2:
            a.set_xlabel("Turn")
fig.supylabel("Accuracy (%)", x=0.02, y=0.5, rotation="vertical", fontsize=24)

# ── 5. 범례 두 줄 ─────────────────────────────────────────
handles, labels = [], []
for a in axes:
    h, l = a.get_legend_handles_labels()
    handles += h; labels += l
hdict = {lab: h for lab, h in zip(labels, handles)}   # ← 올바른 매핑

top    = ["RMA (Oracle Rewrite)", "Baseline"]
bottom = ["RMA (Oracle Rewrite)+", "RMA+", "Baseline+"]

fig.legend([hdict[t] for t in top], top,
           loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.05))
fig.legend([hdict[b] for b in bottom], bottom,
           loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.11))

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()
fig.savefig("overall_comparison_4x3.pdf", bbox_inches="tight")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

# ── 1. 데이터 로드 동일 ──────────────────────────────
df = pd.read_csv("datasets/result/new_all_performance.tsv", sep="\t")
turn_cols, turn_labels = ["turn_2","turn_3","turn_4","turn_5"], [2,3,4,5]
closed = ["o4-mini","GPT-4.1","GPT-4o-mini","Gemini-2.0-flash"]
opens  = ["LLaMA-3.2 (3B)","Phi-4-mini (4B)","Qwen2.5 (3B)","Qwen3 (4B)"]

# ── 2. 팔레트·헬퍼 동일 ──────────────────────────────
cmap = get_cmap("tab10")
palette = {"Baseline":"#bdbdbd","Baseline+":"#525252",
           "Ours":"#9ec2ff","Ours+":cmap(0),
           "RMA":"#ffb55a","RMA+":"#d95f0e"}
pick = lambda m: palette.get(m,"#999999")
def pretty(m):            # 같은 함수
    base = ("RMA (Oracle Rewrite)" if m.startswith("Ours")
            else "Baseline" if m.startswith("Baseline")
            else "RMA" if m.startswith("RMA") else m)
    return base + ("+" if m.endswith("+") else "")

# ── 3. y‑축 범위(상위 두 그룹만) ─────────────────────
mask_top = (df["model"].isin(closed)) | (df["method"].str.endswith("+"))
ymin_top = df.loc[mask_top, turn_cols].min().min() - 1
ymax_top = df.loc[mask_top, turn_cols].max().max() + 1

# ── 4. Figure / GridSpec 2×5  (마지막 칼럼은 범례용) ──
# plt.rcParams.update({"axes.titlesize":24,"axes.labelsize":24,
#                      "xtick.labelsize":14,"ytick.labelsize":20,
#                      "legend.fontsize":24})

plt.rcParams.update({
    "axes.titlesize":16,      # 18 → 16
    "axes.labelsize":16,      # 18 → 16
    "xtick.labelsize":11,     # 12 → 11
    "ytick.labelsize":14,     # 16 → 14
    "legend.fontsize":20      # 18 → 16
})

fig = plt.figure(figsize=(17,6.5))        # 폭↓, 높이↓ (가독성 충분)
gs  = fig.add_gridspec(
        2, 5,
        width_ratios=[1, 1, 1, 1, 0.4],   # ← 범례 칸 0.4 로 살짝 확대
        wspace=0.25, hspace=0.30)         # 서브플롯 간격 약간 줄임

axes = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(4)]
leg_ax = fig.add_subplot(gs[:, 4]); leg_ax.axis("off")  # 범례 전용

plot_order = [("Baseline+","s","--"),("Baseline","s","--"),
              ("Ours+","o","-"),("Ours","o","-"),
              ("RMA+","^",":"),("RMA","^",":")]

seen=set()
def draw(ax, subset, plus_filter=None):
    for m, mk, ls in plot_order:
        if plus_filter is True  and not m.endswith("+"): continue
        if plus_filter is False and  m.endswith("+"):   continue
        row = subset[subset["method"]==m]
        if row.empty: continue
        lab = pretty(m)
        ax.plot(turn_labels, row.iloc[0][turn_cols], marker=mk,
                linestyle=ls, linewidth=2, color=pick(m),
                label=None if lab in seen else lab)
        seen.add(lab)

# Row 0: 클로즈드
for i, m in enumerate(closed):
    a = axes[i]; a.set_title(m)
    draw(a, df[df["model"]==m])
    a.set_ylim(ymin_top, ymax_top); a.grid(axis="y", ls=":", alpha=.4)

# Row 1: 오픈(+)
for i, m in enumerate(opens):
    a = axes[4+i]; a.set_title(f"{m} (+)")
    draw(a, df[(df["model"]==m) & (df["method"].str.endswith("+"))], plus_filter=True)
    a.set_ylim(ymin_top, ymax_top); a.grid(axis="y", ls=":", alpha=.4)

# 공통 축 레이블
for a in axes[4:]: a.set_xlabel("Turn")
#fig.supylabel("Accuracy (%)", x=0.07, y=0.5, rotation="vertical", fontsize=24)
fig.supylabel("Accuracy (%)", x=0.06, y=0.52, rotation="vertical", fontsize=18)
# ── 5. 범례 (leg_ax 위치) ─────────────────────────────
handles, labels = [], []
for a in axes:
    h,l = a.get_legend_handles_labels()
    handles += h; labels += l
from collections import OrderedDict
uniq = OrderedDict(zip(labels, handles))          # 순서 유지 & 중복 제거
# leg_ax.legend(uniq.values(), uniq.keys(),
#               loc="center", frameon=False)
leg_ax.legend(
    uniq.values(), uniq.keys(),
    loc="center left", bbox_to_anchor=(0, 0.5),   # ← 왼쪽 정렬
    frameon=False,
    handlelength=1.8,
    borderpad=0.2,
    labelspacing=0.3
)
fig.savefig("overall_comparison_4x2.pdf", bbox_inches="tight")
plt.show()


In [ ]:
import os, shutil, pandas as pd, matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
from matplotlib.lines import Line2D

# ---------- 데이터 ----------
#df = pd.read_csv("datasets/result/new_all_performance.tsv", sep="\t")
df = pd.read_csv("all_new_performance.tsv", sep="\t")
turn_cols, turn_labels = ["turn_2","turn_3","turn_4","turn_5"], [2,3,4,5]
closed = ["o4-mini","GPT-4.1","GPT-4o-mini","Gemini-2.0-flash"]
opens  = ["LLaMA-3.2 (3B)","Phi-4-mini (4B)","Qwen2.5 (3B)","Qwen3 (4B)"]

# ---------- 팔레트 ----------
cmap = get_cmap("tab10")
palette = {
    "Baseline":  "#4d4d4d",   # non-plus
    "Baseline+": "#000000",   # plus
    "Ours":      "#1e6abe",   # RMA (Oracle)
    "Ours+":     "#66a8ff",   # RMA (Oracle)+
    "RMA":       "#ffb55a",
    "RMA+":      "#d95f0e",
}
pick = lambda m: palette.get(m, "#999999")
def pretty(m):
    base = ("RMA (Oracle Rewrite)" if m.startswith("Ours")
            else "Baseline" if m.startswith("Baseline")
            else "RMA" if m.startswith("RMA") else m)
    return base + ("+" if m.endswith("+") else "")

# ---------- y-축 범위 ----------
mask_top = (df["model"].isin(closed)) | (df["method"].str.endswith("+"))
ymin_top = df.loc[mask_top, turn_cols].min().min() - 1
ymax_top = df.loc[mask_top, turn_cols].max().max() + 1

# ---------- Figure ----------
plt.rcParams.update({
    "axes.titlesize":18, "axes.labelsize":18,
    "xtick.labelsize":13, "ytick.labelsize":16,
    "legend.fontsize":22
})
fig = plt.figure(figsize=(20, 8))
gs  = fig.add_gridspec(2, 5, width_ratios=[1,1,1,1,0.45], wspace=0.28, hspace=0.35)
axes = [fig.add_subplot(gs[r,c]) for r in range(2) for c in range(4)]
leg_ax = fig.add_subplot(gs[:,4]); leg_ax.axis("off")

plot_order = [("Baseline+","s","--"),("Baseline","s","--"),
              ("Ours+","o","-"),("Ours","o","-"),
              ("RMA+","^",":"),("RMA","^",":")]

seen=set()
def draw(ax, subset, plus_filter=None):
    for m, mk, ls in plot_order:
        if plus_filter and not m.endswith("+"): continue
        if plus_filter is False and m.endswith("+"): continue
        row = subset[subset["method"]==m]
        if row.empty: continue
        lab = pretty(m)
        ax.plot(turn_labels, row.iloc[0][turn_cols],
                marker=mk, linestyle=ls, linewidth=2.5,
                color=pick(m), label=None if lab in seen else lab)
        seen.add(lab)

# ---------- 그리기 ----------
for i, m in enumerate(closed):
    ax = axes[i]; ax.set_title(m)
    draw(ax, df[df["model"]==m])
    ax.set_ylim(ymin_top, ymax_top); ax.grid(axis="y", ls=":", alpha=.4)

for i, m in enumerate(opens):
    ax = axes[4+i]; ax.set_title(f"{m} (+)")
    draw(ax, df[(df["model"]==m) & (df["method"].str.endswith("+"))], plus_filter=True)
    ax.set_ylim(ymin_top, ymax_top); ax.grid(axis="y", ls=":", alpha=.4)

for a in axes[4:]: a.set_xlabel("Turn")
fig.supylabel("Accuracy (%)", x=0.055, y=0.52, rotation="vertical", fontsize=20)

# ---------- 범례 ----------
handles, labels = [], []
for a in axes:
    h,l = a.get_legend_handles_labels()
    handles += h; labels += l
uniq = {l:h for h,l in zip(handles, labels)}  # 순서 유지 필요 시 OrderedDict 사용

order = ["Baseline", "RMA (Oracle Rewrite)", "_gap", "_gap",
         "Baseline+", "RMA (Oracle Rewrite)+", "RMA+", "RMA"]
final_h, final_l = [], []
for lab in order:
    if lab == "_gap":
        final_h.append(Line2D([],[], linestyle=""))
        final_l.append("")    # 공백 줄
    elif lab in uniq:
        final_h.append(uniq[lab])
        final_l.append(lab)

leg_ax.legend(final_h, final_l, loc="center left", bbox_to_anchor=(0,0.5),
              frameon=False, handlelength=2.0, borderpad=0.3, labelspacing=0.5)
fig.savefig("overall_comparison_4x2.pdf", bbox_inches="tight")
plt.show()


In [ ]:
import os, shutil, pandas as pd, matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
from matplotlib.lines import Line2D

# ---------- 데이터 ----------
#df = pd.read_csv("datasets/result/new_all_performance.tsv", sep="\t")
df = pd.read_csv("all_new_performance.tsv", sep="\t")
turn_cols, turn_labels = ["turn_2","turn_3","turn_4","turn_5"], [2,3,4,5]
closed = ["o4-mini","GPT-4.1","GPT-5-mini","Gemini-2.0-flash"]
opens  = ["LLaMA-3.2 (3B)","Phi-4-mini (4B)","Qwen3 (4B)", "Qwen2.5 (3B)"]

# ---------- 팔레트 ----------
cmap = get_cmap("tab10")
palette = {
    "Baseline":  "#4d4d4d",   # non-plus
    "Baseline+": "#000000",   # plus
    "Ours":      "#1e6abe",   # RMA (Oracle)
    "Ours+":     "#66a8ff",   # RMA (Oracle)+
    "RMA":       "#ffb55a",
    "RMA+":      "#d95f0e",
}
pick = lambda m: palette.get(m, "#999999")
def pretty(m):
    base = ("RMA (Oracle Rewrite)" if m.startswith("Ours")
            else "Baseline" if m.startswith("Baseline")
            else "RMA" if m.startswith("RMA") else m)
    return base + ("+" if m.endswith("+") else "")

# ---------- y-축 범위 ----------
mask_top = (df["model"].isin(closed)) | (df["method"].str.endswith("+"))
ymin_top = df.loc[mask_top, turn_cols].min().min() - 1
ymax_top = df.loc[mask_top, turn_cols].max().max() + 1

# ---------- Figure ----------
plt.rcParams.update({
    "axes.titlesize":18, "axes.labelsize":18,
    "xtick.labelsize":13, "ytick.labelsize":16,
    "legend.fontsize":22
})
fig = plt.figure(figsize=(20, 8))
gs  = fig.add_gridspec(2, 5, width_ratios=[1,1,1,1,0.45], wspace=0.28, hspace=0.35)
axes = [fig.add_subplot(gs[r,c]) for r in range(2) for c in range(4)]
leg_ax = fig.add_subplot(gs[:,4]); leg_ax.axis("off")

plot_order = [("Baseline+","s","--"),("Baseline","s","--"),
              ("Ours+","o","-"),("Ours","o","-"),
              ("RMA+","^",":"),("RMA","^",":")]

seen=set()
def draw(ax, subset, plus_filter=None):
    for m, mk, ls in plot_order:
        if plus_filter and not m.endswith("+"): continue
        if plus_filter is False and m.endswith("+"): continue
        row = subset[subset["method"]==m]
        if row.empty: continue
        lab = pretty(m)
        ax.plot(turn_labels, row.iloc[0][turn_cols],
                marker=mk, linestyle=ls, linewidth=2.5,
                color=pick(m), label=None if lab in seen else lab)
        seen.add(lab)

# ---------- 그리기 ----------
for i, m in enumerate(closed):
    ax = axes[i]; ax.set_title(m)
    draw(ax, df[df["model"]==m])
    ax.set_ylim(ymin_top, ymax_top); ax.grid(axis="y", ls=":", alpha=.4)

for i, m in enumerate(opens):
    print(i, m)
    ax = axes[4+i]; ax.set_title(f"{m} (+)")
    draw(ax, df[(df["model"]==m) & (df["method"].str.endswith("+"))], plus_filter=True)
    ax.set_ylim(ymin_top, ymax_top); ax.grid(axis="y", ls=":", alpha=.4)

for a in axes[4:]: a.set_xlabel("Turn")
fig.supylabel("Accuracy (%)", x=0.055, y=0.52, rotation="vertical", fontsize=20)

# ---------- 범례 ----------
handles, labels = [], []
for a in axes:
    h,l = a.get_legend_handles_labels()
    handles += h; labels += l
uniq = {l:h for h,l in zip(handles, labels)}  # 순서 유지 필요 시 OrderedDict 사용

order = ["Baseline", "RMA (Oracle Rewrite)", "_gap", "_gap",
         "Baseline+", "RMA (Oracle Rewrite)+", "RMA+"]
final_h, final_l = [], []
for lab in order:
    if lab == "_gap":
        final_h.append(Line2D([],[], linestyle=""))
        final_l.append("")    # 공백 줄
    elif lab in uniq:
        final_h.append(uniq[lab])
        final_l.append(lab)

print('hello')
leg_ax.legend(final_h, final_l, loc="center left", bbox_to_anchor=(0,0.5),
              frameon=False, handlelength=2.0, borderpad=0.3, labelspacing=0.5)
fig.savefig("overall_comparison_4x2_ACL.pdf", bbox_inches="tight")
plt.show()


print(1)

In [ ]:
print(1)

In [ ]:
# 1) 팔레트: 이미 색 정의돼 있으면 그대로, 없다면 추가
colors = {
    "Baseline"   : "#bdbdbd",
    "Baseline+"  : "#525252",
    "RMA"        : "#ffb55a",        # (연주황) non-plus
    "RMA+"       : "#d95f0e",        # ← 진주황 **추가 or 확인**
    "RMA (Oracle Rewrite)"  : "#9ec2ff",
    "RMA (Oracle Rewrite)+": cmap(0),
}

# 2) 열린-소스 막대 순서에 'RMA+' 포함
order_open = [
    "Baseline",
    "Baseline+",
    "RMA",                 # ← non-plus RMA
    "RMA+",
    "RMA (Oracle Rewrite)",
    "RMA (Oracle Rewrite)+",
]

# 3) pivot 함수에 order_open 전달 (코드는 그대로 작동)
open_w   = pivot(opens, order_open)

# 4) 막대폭(bar_w)이 너무 좁아지면 0.12~0.13 로 줄이기
bar_w = 0.12


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

# ── 1. Load ───────────────────────────────────────────────
df = pd.read_csv("datasets/result/new_all_performance.tsv", sep="\t")

turn_cols   = ["turn_2", "turn_3", "turn_4", "turn_5"]
turn_labels = [2, 3, 4, 5]

closed = ["o4-mini", "GPT-4.1", "GPT-4o-mini", "Gemini-2.0-flash"]
opens  = ["LLaMA-3.2 (3B)", "Phi-4-mini (4B)", "Qwen2.5 (3B)", "Qwen3 (4B)"]

# ── 2. Palette ────────────────────────────────────────────
cmap = get_cmap("tab10")          # 0: 파랑, 1: 주황
palette = {
    "Baseline"   : "#bdbdbd",     "Baseline+"  : "#525252",
    "Ours"       : "#9ec2ff",     "Ours+"      : cmap(0),
    "RMA"        : "#ffb55a",     "RMA+"       : "#d95f0e",
}

def pick_color(m):   return palette[m]
def label_of(m):     # 범례용
    if m.startswith("Ours"):     base = "RMA (Oracle Rewrite)"
    elif m.startswith("Baseline"): base = "Baseline"
    else:                        base = "RMA"
    return base + ("+" if m.endswith("+") else "")

# ── 3. y축 범위 ────────────────────────────────────────────
top_mask  = df["model"].isin(closed)
bot_mask  = df["model"].isin(opens)
ymin_top  = df.loc[top_mask, turn_cols].min().min() - 1
ymax_top  = df.loc[top_mask, turn_cols].max().max() + 1
ymin_bot  = df.loc[bot_mask, turn_cols].min().min() - 1
ymax_bot  = df.loc[bot_mask, turn_cols].max().max() + 1

# ── 4. 스타일 ─────────────────────────────────────────────
plt.rcParams.update({
    "axes.titlesize":24, "axes.labelsize":20,
    "xtick.labelsize":14,"ytick.labelsize":14,
    "legend.fontsize":24,
})

fig, axes = plt.subplots(2, 4, figsize=(18, 6.8), sharex=True, sharey=False)
axes = axes.flatten()
ax = lambda r, c: axes[r*4 + c]

legend_seen=set()
order = [("Baseline+","s","--"),("Baseline","s","--"),
         ("Ours+"   ,"o","-"), ("Ours","o","-"),
         ("RMA+"    ,"^",":"), ("RMA","^",":")]

def draw_lines(ax_obj, subset):
    for m, mk, ls in order:
        row = subset[subset["method"] == m]
        if row.empty: continue
        ax_obj.plot(turn_labels, row.iloc[0][turn_cols],
                    marker=mk, linestyle=ls, linewidth=2,
                    color=pick_color(m),
                    label=label_of(m) if label_of(m) not in legend_seen else None)
        legend_seen.add(label_of(m))

# 행0: closed
for c, model in enumerate(closed):
    a = ax(0,c); a.set_title(model)
    draw_lines(a, df[df["model"]==model])
    a.set_ylim(ymin_top, ymax_top); a.grid(axis="y", ls=":", alpha=.4)

# 행1: open (plus+non-plus 함께)
for c, model in enumerate(opens):
    a = ax(1,c); a.set_title(model)
    draw_lines(a, df[df["model"]==model])
    a.set_ylim(ymin_bot, ymax_bot); a.grid(axis="y", ls=":", alpha=.4)

# 축·레이블
for r in range(2):
    for c in range(4):
        axes[r*4+c].set_xticks(turn_labels)
        if r==1: axes[r*4+c].set_xlabel("Turn")
fig.supylabel("Accuracy (%)", x=0.02, y=0.5, rotation="vertical", fontsize=24)

# ── 5. 범례 두 줄 ─────────────────────────────────────────
handles, labels = [], []
for ax_ in axes:
    h,l = ax_.get_legend_handles_labels(); handles+=h; labels+=l
hdict = {lab:h for lab,h in zip(labels, handles)}

top    = ["RMA (Oracle Rewrite)", "Baseline"]
bottom = ["RMA (Oracle Rewrite)+", "RMA+", "Baseline+"]

fig.legend([hdict[t] for t in top], top,
           loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5,-0.04))
fig.legend([hdict[b] for b in bottom], bottom,
           loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5,-0.09))

plt.tight_layout(rect=[0,0.05,1,1])
plt.show()
fig.savefig("overall_comparison_2x4.pdf", bbox_inches="tight")


In [ ]:
# plot_styles.py  ───────────────────────────────────────────────
MODEL_ORDER = [
    #"o4-mini", "GPT-4.1", "GPT-4o-mini", "Gemini-2.0-flash",    
    "o4-mini", "GPT-4.1", "GPT-5-mini", "Gemini-2.0-flash",    
    "LLaMA-3.2 (3B)", "Phi-4-mini (4B)", "Qwen2.5 (3B)", "Qwen3 (4B)",
]

# Paul Tol 8-색 색맹 대응 팔레트
PALETTE = [
    "#E69F00", "#56B4E9", "#009E73", "#D55E00",
    "#F0E442", "#0072B2", "#CC79A7", "#999999",
]

MODEL_COLOR = {m: PALETTE[i] for i, m in enumerate(MODEL_ORDER)}
HATCHES     = ["///", "\\\\", "...", "xxx", "++", "--", "oo", "**"]
MODEL_HATCH = {m: HATCHES[i] for i, m in enumerate(MODEL_ORDER)}

models = [m for m in MODEL_ORDER if m in turn_data["model"].values]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ----------------------- 1. Style presets -----------------------
BIG_TITLE   = 15   # subplot title
AXIS_LABEL  = 20   # x / y label
TICK_LABEL  = 11   # tick label
LEGEND_FONT = 17   # legend
TICK_FONT = 17 # label

plt.rcParams.update({
    "axes.titlesize": BIG_TITLE,
    "axes.labelsize": AXIS_LABEL,
    "xtick.labelsize": TICK_LABEL,
    "ytick.labelsize": TICK_LABEL,
    "legend.fontsize": LEGEND_FONT,
})

# ----------------------- 2. Data prep -----------------------
file_path = "all_new_history_coreference.tsv"
coref_data = pd.read_csv(file_path, sep="\t")

selected_turn   = "Turn4"
turn_distances  = ["coref_turn1", "coref_turn2", "coref_turn3"]
distance_labels = ["1-turn", "2-turn", "3-turn"]

turn_data = coref_data[coref_data["turn"] == selected_turn]

models = [
    #"o4-mini", "GPT-4.1", "GPT-4o-mini", "Gemini-2.0-flash",
    "LLaMA-3.2 (3B)", "Phi-4-mini (4B)", "Qwen2.5 (3B)", "Qwen3 (4B)",
]
models = [m for m in models if m in turn_data["model"].values]

accuracies = turn_data.set_index("model").loc[models, turn_distances].to_numpy()

# Color-blind-friendly palette
palette = [
    "#E69F00", "#56B4E9", "#009E73", "#D55E00",
    "#F0E442", "#0072B2", "#CC79A7", "#999999",
]
colors  = palette[: len(models)]

# Distinct hatch patterns
hatches = ["///", "\\\\", "...", "xxx", "++", "--", "oo", "**"][: len(models)]

# ----------------------- 3. Plot -----------------------
fig, ax = plt.subplots(figsize=(8, 4))

n_models   = len(models)
index      = np.arange(len(distance_labels))
bar_width  = 0.8 / n_models

for i, (model, color, hatch) in enumerate(zip(models, colors, hatches)):
    print(model)
    ax.bar(
        index + i * bar_width,
        accuracies[i],
        bar_width,
        label=model,
        color     = MODEL_COLOR[model],
        hatch     = MODEL_HATCH[model],
        # color=color,
        # hatch=hatch,
        edgecolor="black",
        linewidth=0.3,
    )

#ax.set_title(f"Coreference Accuracy by Distance — {selected_turn}")
ax.set_title("")
ax.set_xlabel("Coreference Accuracy by Distance")
ax.set_ylabel("Accuracy (%)")
ax.tick_params(axis="both", labelsize=TICK_FONT) 
ax.set_xticks(index + bar_width * (n_models - 1) / 2)
ax.set_xticklabels(distance_labels)
ax.set_ylim(60, 85)
ax.grid(axis="y", linestyle=":")

# legend font size는 rcParams에서 이미 적용
ax.legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

# vector PDF 저장
fig.savefig("coref_turn4.pdf", format="pdf", bbox_inches="tight")

"o4-mini", "GPT-4.1", "GPT-4o-mini", "Gemini-2.0-flash",

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ----------------------- 1. Style presets -----------------------
BIG_TITLE   = 15   # subplot title
AXIS_LABEL  = 20   # x / y label
TICK_LABEL  = 11   # tick label
LEGEND_FONT = 17   # legend
TICK_FONT = 17 # label

plt.rcParams.update({
    "axes.titlesize": BIG_TITLE,
    "axes.labelsize": AXIS_LABEL,
    "xtick.labelsize": TICK_LABEL,
    "ytick.labelsize": TICK_LABEL,
    "legend.fontsize": LEGEND_FONT,
})

# ----------------------- 2. Data prep -----------------------
file_path = "all_new_history_coreference.tsv"
coref_data = pd.read_csv(file_path, sep="\t")

selected_turn   = "Turn4"
turn_distances  = ["coref_turn1", "coref_turn2", "coref_turn3"]
distance_labels = ["1-turn", "2-turn", "3-turn"]

turn_data = coref_data[coref_data["turn"] == selected_turn]

models = [
    "o4-mini", "GPT-4.1", "GPT-5-mini", "Gemini-2.0-flash",
    "LLaMA-3.2 (3B)", "Phi-4-mini (4B)", "Qwen2.5 (3B)", "Qwen3 (4B)",
]
models = [m for m in models if m in turn_data["model"].values]

accuracies = turn_data.set_index("model").loc[models, turn_distances].to_numpy()

# Color-blind-friendly palette
palette = [
    "#E69F00", "#56B4E9", "#009E73", "#D55E00",
    "#F0E442", "#0072B2", "#CC79A7", "#999999",
]
colors  = palette[: len(models)]

# Distinct hatch patterns
hatches = ["///", "\\\\", "...", "xxx", "++", "--", "oo", "**"][: len(models)]

# ----------------------- 3. Plot -----------------------
fig, ax = plt.subplots(figsize=(8, 4))

n_models   = len(models)
index      = np.arange(len(distance_labels))
bar_width  = 0.8 / n_models

for i, (model, color, hatch) in enumerate(zip(models, colors, hatches)):
    ax.bar(
        index + i * bar_width,
        accuracies[i],
        bar_width,
        label=model,
        color     = MODEL_COLOR[model],
        hatch     = MODEL_HATCH[model],
        # color=color,
        # hatch=hatch,
        edgecolor="black",
        linewidth=0.3,
    )

#ax.set_title(f"Coreference Accuracy by Distance — {selected_turn}")
ax.set_title("")
ax.set_xlabel("Coreference Accuracy by Distance")
ax.set_ylabel("Accuracy (%)")
ax.tick_params(axis="both", labelsize=TICK_FONT) 
ax.set_xticks(index + bar_width * (n_models - 1) / 2)
ax.set_xticklabels(distance_labels)
ax.set_ylim(60, 85)
ax.grid(axis="y", linestyle=":")

# legend font size는 rcParams에서 이미 적용
ax.legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

# vector PDF 저장
fig.savefig("coref_turn4_ACL.pdf", format="pdf", bbox_inches="tight")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------- 1. Style presets -----------------------------
BIG_TITLE   = 20  # per-subplot title
AXIS_LABEL  = 20   # y-label (공통)
TICK_LABEL  = 11   # y-tick label
LEGEND_FONT = 20   # legend
LABEL_FONT = 11  
plt.rcParams.update({
    "axes.titlesize": BIG_TITLE,
    "axes.labelsize": AXIS_LABEL,
    "xtick.labelsize": TICK_LABEL,
    "ytick.labelsize": TICK_LABEL,
    "legend.fontsize": LEGEND_FONT,
})

# ----------------------------- 2. Data dict -----------------------------
data = {
    'o4-mini':        {'ours_all': 83.9, 'ours_multi': 74.07, 'base_all': 76.24, 'base_multi': 61.89},
    'gpt41':          {'ours_all': 81.71, 'ours_multi': 74.53, 'base_all': 75.14, 'base_multi': 62.62},
    'gpt-4o-mini':    {'ours_all': 79.91, 'ours_multi': 71.87, 'base_all': 72.04, 'base_multi': 57.56},
    'gemini-2.0-flash':{'ours_all': 81.00, 'ours_multi': 71.63, 'base_all': 76.21, 'base_multi': 61.09},
    'llama3-3b':      {'ours_all': 84.43, 'ours_multi': 73.98, 'base_all': 77.13, 'base_multi': 62.18},
    'phi-4-4b':       {'ours_all': 83.08, 'ours_multi': 70.29, 'base_all': 76.00, 'base_multi': 60.87},
    'qwen2.5-3b':     {'ours_all': 82.62, 'ours_multi': 70.30, 'base_all': 72.57, 'base_multi': 56.05},
    'qwen3-4b':       {'ours_all': 82.62, 'ours_multi': 71.12, 'base_all': 75.01, 'base_multi': 59.74},
}

order = ['o4-mini', 'gpt41', 'gpt-4o-mini', 'gemini-2.0-flash',
         'llama3-3b', 'phi-4-4b', 'qwen2.5-3b', 'qwen3-4b']

order = [
    "o4-mini", "GPT-4.1", "GPT-4o-mini", "Gemini-2.0-flash",
    "LLaMA-3.2 (3B)", "Phi-4-mini (4B)", "Qwen2.5 (3B)", "Qwen3 (4B)",
]

colors = {
    'ours_all':  '#9ecbf7',
    'base_all':  '#f4a4a6',
    'ours_multi':'#3587e4',
    'base_multi':'#d1495b'
}

# ----------------------------- 3. Figure -----------------------------
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
bw = 0.18

for idx, model in enumerate(order):
    ax = axes[idx // 4, idx % 4]
    ours_all, ours_multi = data[model]['ours_all'], data[model]['ours_multi']
    base_all, base_multi = data[model]['base_all'], data[model]['base_multi']
    
    pos  = np.array([-1.5, -0.5, 0.5, 1.5]) * bw
    vals = [ours_all, base_all, ours_multi, base_multi]
    keys = ['ours_all', 'base_all', 'ours_multi', 'base_multi']
    
    bars = ax.bar(pos, vals, bw, color=[colors[k] for k in keys],
                  edgecolor='black', linewidth=0.3, zorder=2)
    
    # annotate bar heights
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.1f}', xy=(bar.get_x() + bw/2, h),
                    xytext=(0, 2), textcoords='offset points',
                    ha='center', va='bottom', fontsize=LABEL_FONT)  # 살짝 키움
    
    # Bracket (Overall)
    x1, x2   = pos[0], pos[1]
    gap_all  = ours_all - base_all
    top_all  = max(ours_all, base_all) + 1.0
    ax.plot([x1, x1, x2, x2], [top_all-0.5, top_all, top_all, top_all-0.5],
            color='black', linewidth=1.2, zorder=3)
    ax.annotate(f'+{gap_all:.1f}', xy=((x1+x2)/2, top_all + 0.3),
                ha='center', va='bottom', fontsize=9, color='red', fontweight='bold')
    
    # Bracket (Multi-turn)
    x1m, x2m = pos[2], pos[3]
    gap_multi  = ours_multi - base_multi
    top_multi  = max(ours_multi, base_multi) + 1.0
    ax.plot([x1m, x1m, x2m, x2m], [top_multi-0.5, top_multi, top_multi, top_multi-0.5],
            color='black', linewidth=1.2, linestyle='dotted', zorder=3)
    ax.annotate(f'+{gap_multi:.1f}', xy=((x1m+x2m)/2, top_multi + 0.3),
                ha='center', va='bottom', fontsize=9, color='red', fontweight='bold')
    
    ax.set_title(model, pad=6)     # BIG_TITLE 적용
    ax.set_xticks([])
    ax.set_ylim(50, 90)
    ax.grid(axis='y', linestyle='--', alpha=0.3, zorder=0)

# 공통 y-label
fig.text(0.04, 0.5, 'Accuracy (%)', va='center', rotation='vertical', fontsize=AXIS_LABEL)

# Legend
handles = [plt.Rectangle((0,0),1,1,color=colors[k]) for k in
           ['ours_all', 'base_all', 'ours_multi','base_multi']]
labels  = ['Ours – Overall', 'Baseline – Overall', 'Ours – Multi-turn Intensive', 'Baseline – Multi-turn Intensive']
fig.legend(handles, labels, loc='lower center',
           ncol=4, bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.tight_layout(rect=[0.05, 0.06, 1, 0.95])
plt.show()

# PDF 저장
fig.savefig("frequent_multiturn.pdf", format="pdf", bbox_inches="tight")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------- 1. Style presets -----------------------------
# 전역 글꼴 크기
BIG_TITLE   = 23   # per-subplot title
AXIS_LABEL  = 23   # 공통 y-label
TICK_LABEL  = 11   # tick labels
LEGEND_FONT = 20 # legend
VALUE_FONT  = 11   # 막대 위 숫자
GAP_FONT    = 23   # +Δ 레이블

plt.rcParams.update({
    "axes.titlesize": BIG_TITLE,
    "axes.labelsize": AXIS_LABEL,
    "xtick.labelsize": TICK_LABEL,
    "ytick.labelsize": TICK_LABEL,
    "legend.fontsize": LEGEND_FONT,
})

# ----------------------------- 2. 데이터 -----------------------------
data = {
    'o4-mini':        {'ours_all': 83.9, 'ours_multi': 74.07, 'base_all': 76.24, 'base_multi': 61.89},
    'GPT-4.1':          {'ours_all': 81.71, 'ours_multi': 74.53, 'base_all': 75.14, 'base_multi': 62.62},
    'GPT-4o-mini':    {'ours_all': 79.91, 'ours_multi': 71.87, 'base_all': 72.04, 'base_multi': 57.56},
    'Gemini-2.0-flash':{'ours_all': 81.00, 'ours_multi': 71.63, 'base_all': 76.21, 'base_multi': 61.09},
    'LLaMA-3.2 (3B)':      {'ours_all': 84.43, 'ours_multi': 73.98, 'base_all': 77.13, 'base_multi': 62.18},
    'Phi-4-mini (4B)':       {'ours_all': 83.08, 'ours_multi': 70.29, 'base_all': 76.00, 'base_multi': 60.87},
    'Qwen2.5 (3B)':     {'ours_all': 82.62, 'ours_multi': 70.30, 'base_all': 72.57, 'base_multi': 56.05},
    'Qwen3 (4B)':       {'ours_all': 82.62, 'ours_multi': 71.12, 'base_all': 75.01, 'base_multi': 59.74},
}

order = ['o4-mini', 'GPT-4.1', 'gpt-4o-mini', 'gemini-2.0-flash',
         'llama3-3b', 'phi-4-mini-4b', 'qwen2.5-3b', 'qwen3-4b']

order = [
    "o4-mini", "GPT-4.1", "GPT-4o-mini", "Gemini-2.0-flash",
    "LLaMA-3.2 (3B)", "Phi-4-mini (4B)", "Qwen2.5 (3B)", "Qwen3 (4B)",
]

colors = {
    'ours_all':  '#9ecbf7',
    'base_all':  '#f4a4a6',
    'ours_multi':'#3587e4',
    'base_multi':'#d1495b'
}

# ----------------------------- 3. Figure -----------------------------
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
#fig.subplots_adjust(wspace=0, hspace=0.2)
bw = 0.18

# y축 범위 (필요하면 자동 계산 가능)
ymin, ymax = 50, 95
fig.text(0.04, 0.5, 'Accuracy (%)', va='center', rotation='vertical', fontsize=AXIS_LABEL)

# 비율 기반 오프셋 (전체 높이 대비 %)
offset_ratio  = 0.025         # +Δ 텍스트와 막대 사이 간격
bracket_ratio = 0.015         # bracket 윗공간

for idx, model in enumerate(order):
    ax = axes[idx // 4, idx % 4]
    ours_all   = data[model]['ours_all']
    base_all   = data[model]['base_all']
    ours_multi = data[model]['ours_multi']
    base_multi = data[model]['base_multi']
    
    pos  = np.array([-1.5, -0.5, 0.5, 1.5]) * bw
    vals = [ours_all, base_all, ours_multi, base_multi]
    keys = ['ours_all', 'base_all', 'ours_multi', 'base_multi']
    
    bars = ax.bar(pos, vals, bw, color=[colors[k] for k in keys],
                  edgecolor='black', linewidth=0.3, zorder=2)
    
    # ---------- 막대 위 숫자 ----------
    for bar in bars:
        h = bar.get_height()
        # ax.annotate(f'{h:.1f}',
        #             xy=(bar.get_x() + bw/2, h),
        #             xytext=(0, 2),
        #             textcoords='offset points',
        #             ha='center', va='bottom',
        #             fontsize=VALUE_FONT)
    
    # ---------- Overall 비교 (+Δ) ----------
    x1, x2     = pos[0], pos[1]
    gap_all    = ours_all - base_all
    top_all    = max(ours_all, base_all) + (ymax - ymin) * bracket_ratio
    ax.plot([x1, x1, x2, x2],
            [top_all - (ymax - ymin)*0.005, top_all, top_all, top_all - (ymax - ymin)*0.005],
            color='black', linewidth=1.2, zorder=3)
    ax.annotate(f'+{gap_all:.1f}',
                xy=((x1 + x2)/2, top_all + (ymax - ymin) * offset_ratio),
                ha='center', va='bottom',
                fontsize=GAP_FONT, color='red', fontweight='bold')
    
    # ---------- Multi-turn 비교 (+Δ) ----------
    x1m, x2m   = pos[2], pos[3]
    gap_multi  = ours_multi - base_multi
    top_multi  = max(ours_multi, base_multi) + (ymax - ymin) * bracket_ratio
    ax.plot([x1m, x1m, x2m, x2m],
            [top_multi - (ymax - ymin)*0.005, top_multi, top_multi, top_multi - (ymax - ymin)*0.005],
            color='black', linewidth=1.2, linestyle='dotted', zorder=3)
    ax.annotate(f'+{gap_multi:.1f}',
                xy=((x1m + x2m)/2, top_multi + (ymax - ymin) * offset_ratio),
                ha='center', va='bottom',
                fontsize=GAP_FONT, color='red', fontweight='bold')
    
    ax.set_title(model, pad=6)
    ax.set_xticks([])
    ax.set_ylim(ymin, ymax)
    ax.grid(axis='y', linestyle='--', alpha=0.3, zorder=0)

# ----------------------------- 4. Legend & Layout -----------------------------
handles = [plt.Rectangle((0,0),1,1,color=colors[k]) for k in
           ['ours_all', 'base_all', 'ours_multi','base_multi']]
labels  = ['Ours – Overall', 'Baseline – Overall', 'Ours – Multi-turn Intensive', 'Baseline – Multi-turn Intensive']

fig.legend(
    handles, labels,
    loc='lower center',
    ncol=2,                      # 2 × 2 배열
    bbox_to_anchor=(0.5, -0.06), # y 값을 −0.12 정도로 내려 범례 높이 확보
    frameon=False,
    columnspacing=1.0,           # (선택) 컬럼 간 간격 조정
    handletextpad=0.4            # (선택) 심볼과 텍스트 간격
)
plt.tight_layout(rect=[0.06, 0.06, 1, 0.95])
plt.show()

# 저장
fig.savefig("frequent_multiturn.pdf", format="pdf", bbox_inches="tight")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------- 1. Style presets -----------------------------
BIG_TITLE   = 22   # per-subplot title
AXIS_LABEL  = 20   # 공통 y-label
TICK_LABEL  = 20   # tick labels
LEGEND_FONT = 20   # legend
VALUE_FONT  = 20   # 막대 위 숫자 (사용 안 해도 일관 유지)
GAP_FONT    = 25   # +Δ annotation

plt.rcParams.update({
    "axes.titlesize": BIG_TITLE,
    "axes.labelsize": AXIS_LABEL,
    "xtick.labelsize": TICK_LABEL,
    "ytick.labelsize": TICK_LABEL,
    "legend.fontsize": LEGEND_FONT,
})

# ----------------------------- 2. 데이터 -----------------------------
data = {
    'Qwen2.5 (3B)': {'ours_all': 82.62, 'ours_multi': 70.30,
                   'base_all': 72.57, 'base_multi': 56.05},
    'Qwen3 (4B)':   {'ours_all': 82.62, 'ours_multi': 71.12,
                   'base_all': 75.01, 'base_multi': 59.74},
}
order  = ['Qwen2.5 (3B)', 'Qwen3 (4B)']
colors = {'ours_all':'#9ecbf7', 'base_all':'#f4a4a6',
          'ours_multi':'#3587e4', 'base_multi':'#d1495b'}

# ----------------------------- 3. Figure -----------------------------
n_models = len(order)
fig, axes = plt.subplots(
    nrows=1, ncols=n_models,
    figsize=(5 * n_models, 5),   # 10×5 inch
    sharey=True
)
axes = np.atleast_1d(axes)
bw = 0.18
ymin, ymax = 50, 95

fig.text(0.04, 0.5, 'Accuracy (%)', va='center',
         rotation='vertical', fontsize=AXIS_LABEL)

offset_ratio, bracket_ratio = 0.03, 0.02   # 비율도 약간 키움

for idx, model in enumerate(order):
    ax = axes[idx]
    ours_all   = data[model]['ours_all'];   base_all   = data[model]['base_all']
    ours_multi = data[model]['ours_multi']; base_multi = data[model]['base_multi']

    pos  = np.array([-1.5, -0.5, 0.5, 1.5]) * bw
    vals = [ours_all, base_all, ours_multi, base_multi]
    keys = ['ours_all', 'base_all', 'ours_multi', 'base_multi']

    ax.bar(pos, vals, bw,
           color=[colors[k] for k in keys],
           edgecolor='black', linewidth=0.4, zorder=2)

    # ----- Δ annotations -----
    def draw_gap(x1, x2, top, gap):
        ax.plot([x1, x1, x2, x2],
                [top-(ymax-ymin)*0.006, top, top, top-(ymax-ymin)*0.006],
                color='black', linewidth=1.3, linestyle='solid', zorder=3)
        ax.annotate(f'+{gap:.1f}',
                    xy=((x1+x2)/2, top + (ymax-ymin)*offset_ratio),
                    ha='center', va='bottom',
                    fontsize=GAP_FONT, color='red', fontweight='bold')

    # -- Overall Δ
    draw_gap(pos[0], pos[1],
             max(ours_all, base_all) + (ymax-ymin)*bracket_ratio,
             ours_all - base_all)
    
    # -- Multi-turn Δ  (점선 → 실선으로 동일 적용)
    draw_gap(pos[2], pos[3],
             max(ours_multi, base_multi) + (ymax-ymin)*bracket_ratio,
             ours_multi - base_multi)

    ax.set_title(model, pad=4)
    ax.set_xticks([])
    ax.set_ylim(ymin, ymax)
    ax.grid(axis='y', linestyle='--', alpha=0.25, zorder=0)

# ----------------------------- 4. Legend & Layout -----------------------------
handles = [plt.Rectangle((0,0),1,1,color=colors[k])
           for k in ['ours_all', 'base_all', 'ours_multi', 'base_multi']]
labels  = ['Ours – Overall', 'Baseline – Overall',
           'Ours – Multi-turn Intensive', 'Baseline – Multi-turn Intensive']

fig.legend(handles, labels,
           loc='lower center', ncol=2,
           bbox_to_anchor=(0.5, -0.10),
           frameon=False, columnspacing=1.2, handletextpad=0.5)

plt.tight_layout(rect=[0.06, 0.08, 1, 0.94])  # rect y-여백도 상황에 맞게
plt.show()

fig.savefig("frequent_multiturn_qwen_only.pdf",
            format="pdf", bbox_inches="tight")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ----------------------------- 1. Style presets -----------------------------
BIG_TITLE   = 22   # per-subplot title
AXIS_LABEL  = 20   # shared y-label
TICK_LABEL  = 20   # tick labels
LEGEND_FONT = 20   # legend
GAP_FONT    = 25   # +Δ annotation

plt.rcParams.update({
    "axes.titlesize": BIG_TITLE,
    "axes.labelsize": AXIS_LABEL,
    "xtick.labelsize": TICK_LABEL,
    "ytick.labelsize": TICK_LABEL,
    "legend.fontsize": LEGEND_FONT,
})

# ----------------------------- 2. Data -----------------------------
data = {
    'Qwen2.5 (3B)': {'ours_all': 82.62, 'ours_multi': 70.30,
                     'base_all': 72.57, 'base_multi': 56.05},
    'Qwen3 (4B)':   {'ours_all': 82.62, 'ours_multi': 71.12,
                     'base_all': 75.01, 'base_multi': 59.74},
}
order = ['Qwen2.5 (3B)', 'Qwen3 (4B)']

# 색상 2종
color_map = {
    'ours_all':   '#3587e4',   # 파랑
    'ours_multi': '#3587e4',
    'base_all':   '#d1495b',   # 빨강
    'base_multi': '#d1495b',
}

# 해치 패턴 (전체=실선, 멀티턴=///)
hatch_map = {
    'ours_all':   '',
    'ours_multi': '///',
    'base_all':   '',
    'base_multi': '///',
}

# ----------------------------- 3. Figure -----------------------------
n_models = len(order)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5), sharey=True)
axes = np.atleast_1d(axes)

bw = 0.18                 # bar width
ymin, ymax = 50, 95
offset_ratio, bracket_ratio = 0.03, 0.02  # gap annotation 위치

fig.text(0.04, 0.5, 'Accuracy (%)', va='center',
         rotation='vertical', fontsize=AXIS_LABEL)

for idx, model in enumerate(order):
    ax = axes[idx]
    ours_all   = data[model]['ours_all']
    base_all   = data[model]['base_all']
    ours_multi = data[model]['ours_multi']
    base_multi = data[model]['base_multi']

    pos  = np.array([-1.5, -0.5, 0.5, 1.5]) * bw
    vals = [ours_all, base_all, ours_multi, base_multi]
    keys = ['ours_all', 'base_all', 'ours_multi', 'base_multi']

    # 막대 그리기
    for p, v, k in zip(pos, vals, keys):
        ax.bar(p, v, bw,
               color=color_map[k],
               hatch=hatch_map[k],
               edgecolor='black', linewidth=0.4, zorder=2)

    # Δ 표시 함수
    def draw_gap(x1, x2, top, gap):
        ax.plot([x1, x1, x2, x2],
                [top-(ymax-ymin)*0.006, top, top, top-(ymax-ymin)*0.006],
                color='black', linewidth=1.3, zorder=3)
        ax.annotate(f'+{gap:.1f}',
                    xy=((x1+x2)/2, top + (ymax-ymin)*offset_ratio),
                    ha='center', va='bottom',
                    fontsize=GAP_FONT, color='red', fontweight='bold')

    draw_gap(pos[0], pos[1],
             max(ours_all, base_all) + (ymax-ymin)*bracket_ratio,
             ours_all - base_all)
    draw_gap(pos[2], pos[3],
             max(ours_multi, base_multi) + (ymax-ymin)*bracket_ratio,
             ours_multi - base_multi)

    ax.set_title(model, pad=4)
    ax.set_xticks([])
    ax.set_ylim(ymin, ymax)
    ax.grid(axis='y', linestyle='--', alpha=0.25, zorder=0)

# ----------------------------- 4. Legend -----------------------------
handles = [
    mpatches.Patch(facecolor='#3587e4', edgecolor='black', hatch='',    label='Ours – Overall'),
    mpatches.Patch(facecolor='#d1495b', edgecolor='black', hatch='',    label='Baseline – Overall'),
    mpatches.Patch(facecolor='#3587e4', edgecolor='black', hatch='///', label='Ours – Multi-turn'),
    mpatches.Patch(facecolor='#d1495b', edgecolor='black', hatch='///', label='Baseline – Multi-turn'),
]

fig.legend(handles=handles,
           loc='lower center', ncol=2,
           bbox_to_anchor=(0.5, -0.10),
           frameon=False, columnspacing=1.2, handletextpad=0.5)

plt.tight_layout(rect=[0.06, 0.08, 1, 0.94])
plt.show()

fig.savefig("frequent_multiturn_qwen_only_hatched_legend.pdf",
            format="pdf", bbox_inches="tight")
